# World Cup 2026 — Match & Tournament Prediction (results-based)

Predict 2026 World Cup match outcomes and tournament-level probabilities from **international
match results only** (goals + fixtures) — no xG or player-level data. This deliberately trades
feature richness for sample size, avoiding the ~280-match data ceiling that constrained the
previous StatsBomb-based project.

The data is `martj42/international_results`: **49,496** internationals from 1872 to present
(`date, home_team, away_team, home_score, away_score, tournament, city, country, neutral`), plus
`shootouts.csv` (for knockout resolution) and `goalscorers.csv` (minute-level, optional). Because
the raw columns are minimal, almost every useful feature is **derived** — team-strength ratings,
recent form, tournament importance, venue context — all computed strictly as-of the match date to
avoid lookahead.

## Modeling plan
- **Poisson / Dixon-Coles goals model is the spine.** The 2026 format resolves on **goal
  difference** (group-stage tiebreakers and picking the 8 best third-placed teams), and the Monte
  Carlo simulator samples scorelines — so we need goals, not just W/D/L. Win/draw/loss
  probabilities fall out of the Poisson PMF for free. Dixon-Coles adds the low-score correlation
  correction and time-weighting.
- **Then a direct W/D/L classifier** on the same features, tested against the PMF-derived
  probabilities on time-ordered holdout — sequenced after the spine, not built in parallel. (This
  was rejected at 280 matches last project; with ~49k it gets a fair re-test.)
- **Monte Carlo tournament simulator** driven by the combined λ's.
- **Evaluation:** RPS, log loss, Brier — calibration-first (the simulator compounds per-match
  error across rounds). **Walk-forward validation only** (no random K-fold — leakage). Baselines
  to beat: Elo and bookmaker closing odds.

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load the three raw martj42 files: match results (the main modeling table),
# goal-level data, and penalty shootouts. .info() surfaces dtypes + null counts.
results = pd.read_csv('../data/raw/results.csv')
goalscorers = pd.read_csv('../data/raw/goalscorers.csv')
shootouts = pd.read_csv('../data/raw/shootouts.csv')

print(results.info())
print(results.shape)
print('\n')
print(goalscorers.info())
print(goalscorers.shape)
print('\n')
print(shootouts.info())
print(shootouts.shape)

### What the initial load shows

- **49,496 matches**, 9 columns. `neutral` parses cleanly as `bool` (no misalignment despite
  quoted city names like `"Washington, D.C."`).
- **12 matches have missing scores** (`home_score`/`away_score` = 49,484 non-null) — likely
  unplayed or abandoned fixtures; drop or handle before modeling.
- `goalscorers` and `shootouts` load fine; `shootouts.first_shooter` is mostly null (expected —
  rarely recorded).

Distribution checks (below) surface the decisions that shape data scope:

- **Friendlies dominate** — ~37% of all matches (18,388). Low-stakes, rotated squads, often
  lopsided → candidates for down-weighting rather than equal weight.
- **Qualifier-heavy**: World Cup / Euro / AFCON qualifiers make up the bulk; actual tournament
  *finals* are a small slice, and many qualifiers are giant-vs-minnow blowouts.
- **Front-loaded to the modern era**: ~32k matches are from 1990 on, ~4k before 1950. Once
  recency-weighted, the pre-war era contributes almost nothing — the effective sample is modern.
- **200 distinct tournaments** → need an importance *tiering*, not 200 dummy variables.
- Domain prior: the same ~6 nations win almost every World Cup (Brazil, Argentina, Germany,
  Italy, England, France) → the team-strength rating carries most of the signal; the model's job
  is to rank favorites and stay calibrated, not to pick upsets.

### Cleaning `results`

- **Drop the 12 no-result rows** — unplayed/abandoned fixtures have no scoreline to model; imputing one would invent data.
- **Cast goals to int** — they were `float64` only because those NaNs forced it. Integer goals keep the simulator's goal-difference arithmetic clean.
- **Parse `date` → datetime and sort chronologically** — the backbone of all no-lookahead / walk-forward work; a clean `0..N` index means time-ordered splits are simple slices.
- **`goalscorers` / `shootouts`**: convert `date` for consistency, but leave their nulls (informative gaps, not corruption) and don't dedupe `goalscorers` — its repeats are real braces.

In [3]:
print(results['home_score'].isna().sum())
print(results['away_score'].isna().sum())

# Drop matches with no result
results = results.dropna(subset=['home_score', 'away_score']).copy()

# Convert home_score & away_score to integers
results[['home_score', 'away_score']] = results[['home_score', 'away_score']].astype(int)

# Convert date to datetime
results['date'] = pd.to_datetime(results['date'])
results = results.sort_values('date').reset_index(drop=True)

print(results.shape) # Rows dropped 
print(results.dtypes[['home_score', 'away_score', 'date']]) # All converted

12
12
(49484, 9)
home_score             int32
away_score             int32
date          datetime64[us]
dtype: object


In [4]:
goalscorers['date'] = pd.to_datetime(goalscorers['date'])
print(goalscorers['date'].dtype) # Converted

shootouts['date'] = pd.to_datetime(shootouts['date'])
print(shootouts['date'].dtype) # Converted

datetime64[us]
datetime64[us]


In [ ]:
# Exact-duplicate rows (all columns identical)
print(results.duplicated().sum())
print(goalscorers.duplicated().sum())   # 79 expected: same scorer twice with missing/equal minute = real braces, not errors
print(shootouts.duplicated().sum())

# Same MATCH logged twice: identical date + teams, even if other columns differ
key = ['date', 'home_team', 'away_team']
print(results.duplicated(subset=key).sum())

# Inspect both copies of each offending pair before deciding (keep=False marks all duplicates)
results[results.duplicated(subset=key, keep=False)].sort_values(key)

# Drop the 2 match-key dupes: 1 true dupe (Gibraltar, venue spelt two ways) + 1 ambiguous 1974 friendly
results = results.drop_duplicates(subset=key, keep='first').reset_index(drop=True)
print(results.shape)

### Duplicate check

- **`results` / `shootouts`: no exact-duplicate rows.**
- **`goalscorers`: 79 exact duplicates — left as-is.** These are real braces (same scorer twice in a match with missing/equal minute), not errors; deleting them would drop actual goals.
- **`results`: 2 match-key duplicates** (same date + teams), both dropped:
  - *Gibraltar 4–1 Cayman Islands (2026)* — a true duplicate; the venue was entered as both "Europa Point" and "Gibraltar" (same place).
  - *Tahiti vs New Caledonia (1974)* — ambiguous (opposite scores same day), but a pre-1990 minnow friendly with near-zero weight, so harmless to drop.

→ **49,482 clean matches** to work from.